# Entornos de Transformación Digital: Modelado Predictivo y Clasificación Estratégica con el Dataset House Prices

> En la gestión estratégica de la transformación digital, las organizaciones se enfrentan constantemente a la disyuntiva entre **estimar un valor monetario continuo** (¿Cuánto costará este activo inmobiliario?) y **tomar una decisión categórica de negocio** (¿Es este inmueble de segmento *Premium* o *Estándar*? ¿Representa una oportunidad de alta rentabilidad o alto riesgo?).

En este laboratorio resolveremos de forma articulada:
1. **Regresión Lineal Múltiple (`Multiple Linear Regression`):** Para la estimación cuantitativa continua de precios y valoración de activos (*Pricing Models*).
2. **Transformación Estratégica de Regresión a Clasificación:** Formulación de reglas de segmentación y categorización de mercado a partir del precio de venta.
3. **Clasificación con Regresión Logística (`Logistic Regression`):** Modelo paramétrico probabilístico para evaluar la pertenencia a segmentos estratégicos (*Targeting / Scoring*).
4. **Clasificadores Adecuados para Datos Tabulares Complejos:** 
   * **`Random Forest Classifier (Ensemble)`:** Árboles de decisión en ensamble para capturar no-linealidades y mitigar el sobreajuste.
   * **`Gradient Boosting / HistGradientBoosting:`** El estándar de la industria en datos tabulares para maximizar la capacidad predictiva en entornos competitivos.
5. **Evaluación de Impacto de Negocio:** Matriz de confusión, curvas ROC-AUC, precisión, exhaustividad (*Recall*) y evaluación del costo de falsos positivos y falsos negativos.

> **Referencia estratégica:** El [mapa de algoritmos de Aprendizaje Supervisado de Scikit-Learn](https://scikit-learn.org/stable/supervised_learning.html) orienta a los involucrados en la selección del algoritmo e clasificación adecuado según el volumen de datos y el tipo de resultado esperado. 

## 1. Ingesta de Datos y Gobierno del Pipeline (*Data Ingestion*)

> Cargamos los datos del mercado inmobiliario (`House Prices`). El dataset se carga en un archivo local `house_train.csv` o el dataset oficial esta disponible en Kaggle.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Intento de carga local o descarga remota
data_path = '../datasets/house_train.csv'
housing = pd.read_csv(data_path, index_col=0)

# Ingeniería de atributos inicial: Cálculo de antigüedad del inmueble (Time-to-Market)
housing['Age'] = housing['YrSold'].astype(float) - housing['YearBuilt'].astype(float)

print(f"Dimensiones iniciales del repositorio: {housing.shape[0]} registros y {housing.shape[1]} columnas")
housing[['Neighborhood', 'YearBuilt', 'YrSold', 'Age', 'SalePrice']].head()

## 2. Limpieza Estratégica y Preprocesamiento (*Data Curation*)

> Para garantizar la validez estadística en las zonas urbanas analizadas (`Neighborhood`), filtramos vecindarios con representatividad muestral ($>30$ transacciones) y codificamos las variables categóricas seleccionadas con la técnica `One-Hot Encoding`.

In [ ]:
# 1. Filtrado de vecindarios con representatividad estadística (> 30 observaciones)
counts = housing['Neighborhood'].value_counts()
more_than_30 = list(counts[counts > 30].index)
housing = housing.loc[housing['Neighborhood'].isin(more_than_30)].copy()

# 2. Definición directa de variables predictoras clave
base_features = ['LotArea', 'OverallQual', 'OverallCond', '1stFlrSF', '2ndFlrSF', 'BedroomAbvGr', 'Age', 'CentralAir']

# 3. Codificación robusta de Aire Acondicionado Central (soporta texto, categorías o numérico)
if 'CentralAir' in housing.columns:
    housing['CentralAir'] = housing['CentralAir'].map({'N': 0, 'Y': 1, 0: 0, 1: 1}).fillna(0).astype(int)

# 4. Limpieza de columnas dummies previas si la celda se re-ejecuta
cols_dummies_previas = [c for c in housing.columns if c.startswith('Nb_')]
if cols_dummies_previas:
    housing = housing.drop(columns=cols_dummies_previas)

# 5. One-Hot Encoding especificando dtype=int para evitar problemas con booleanos
dummies_nb = pd.get_dummies(housing['Neighborhood'], prefix='Nb', drop_first=True, dtype=int)
housing = pd.concat([housing, dummies_nb], axis=1)

# 6. Lista consolidada de características sin duplicados
all_features = base_features + list(dummies_nb.columns)

# 7. Imputación de baselines y conversión a arrays numéricos (float)
housing[all_features] = housing[all_features].apply(pd.to_numeric, errors='coerce')
housing[all_features] = housing[all_features].fillna(housing[all_features].median())

X_continuous = housing[all_features].values.astype(float)
y_continuous = housing['SalePrice'].values.astype(float)

print(f"Total de características operacionales listas para modelado: {len(all_features)}")


In [ ]:
print(f"{all_features}")

## 3. Regresión Lineal Múltiple (*Multiple Linear Regression*)

> ¿Por qué se considera un modelo de `Multiple Linear Regression`? La regresión lineal modela un **resultado numérico continuo** (el precio exacto en dólares), por lo que formalmente es una técnica de **regresión**, no de clasificación.

> Comparamos la regresión lineal contra el **modelo de línea base o punto de referencia (`Baseline / Benchmark`)**, el cual se limita a predecir la media del mercado ($180,000 aprox.). Esta comparación demuestra a los stakeholders el valor económico agregado (*ROI*) del algoritmo al reducir el margen de error.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Partición estratégica de datos (80% Entrenamiento y 20% validación del modelo de negocio)
X_tr_reg, X_te_reg, y_tr_reg, y_te_reg = train_test_split(
    X_continuous, y_continuous, test_size=0.20, random_state=42
)

# 1. Modelo Baseline (Predecir la media de entrenamiento)
pred_baseline = np.full_like(y_te_reg, y_tr_reg.mean())
rmse_baseline = np.sqrt(mean_squared_error(y_te_reg, pred_baseline))

# 2. Regresión Lineal Múltiple
modelo_regresion = LinearRegression()
modelo_regresion.fit(X_tr_reg, y_tr_reg)
pred_regresion = modelo_regresion.predict(X_te_reg)
rmse_regresion = np.sqrt(mean_squared_error(y_te_reg, pred_regresion))
r2_regresion = r2_score(y_te_reg, pred_regresion)

print("Impacto financiero del modelo de Regresión Múltiple")
print(f"Error Medio Modelo Baseline (RMSE Baseline): ${rmse_baseline:,.2f}")
print(f"Error Medio Regresión Lineal (RMSE Modelo):   ${rmse_regresion:,.2f}")
print(f"Reducción del error en la valoración de activos: {((rmse_baseline - rmse_regresion)/rmse_baseline):.2%}")
print(f"Capacidad explicativa de la varianza del mercado (R^2 Score): {r2_regresion:.2%}")

> - Reduce el riesgo de valoración en un 53.47% respecto a estimaciones simplistas.
> - Explica el 78.35% de la dinámica de precios del mercado evaluado.
> - Establece la base cuantitativa para la siguiente fase del laboratorio: la clasificación y segmentación de propiedades (Segmento Premium vs. Estándar).

## 4. Formulación Estratégica del Problema de Clasificación

> **Del Dato Continuo a la Decisión Categórica**
> En los comités de inversión y planeación estratégica, las decisiones suelen tomarse en términos de portafolios y segmentos de mercado:
- **Segmento 1 (Inmueble de Alta Gama / Premium):** Propiedades con precio de venta en el $25\%$ superior del mercado ($Q_3 \ge \text{P75}$). Requieren estrategias comerciales diferenciadas, canales digitales especializados y esquemas de crédito premium.
- **Segmento 0 (Inmueble Estándar / Accesible):** Propiedades en el $75\%$ inferior del mercado.
> Definimos esta variable binaria (`IsPremium`) para estructurar los algoritmos de clasificación supervisada.

In [ ]:
housing['SalePrice'].describe()

In [ ]:
# Umbral del Percentil 75 para segmentación de portafolio
umbral_premium = np.percentile(y_continuous, 75) # 213437
housing['IsPremium'] = (housing['SalePrice'] >= umbral_premium).astype(int)

y_class = housing['IsPremium'].values
distribucion = housing['IsPremium'].value_counts(normalize=True)

print(f"Umbral de corte financiero para Segmento Premium: ${umbral_premium:,.2f}")
print(f"Distribución del portafolio: {distribucion[0]:.1%} Estándar (Clase 0) | {distribucion[1]:.1%} Premium (Clase 1)")

## 5. Estandarización y Partición Estratégica para Clasificación

> Para garantizar que los modelos lineales y basados en gradientes no se sesguen por atributos con escalas dispares (como `LotArea` frente a `OverallQual`), estandarizamos las variables numéricas.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Partición estratificada para mantener la proporción 80/20 en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X_continuous, y_class, test_size=0.20, random_state=42, stratify=y_class
)

# Estandarización de características (Media 0, Desviación 1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Muestra de entrenamiento: {X_train.shape[0]} inmuebles")
print(f"Muestra de validación directiva: {X_test.shape[0]} inmuebles")

## 6. Clasificación con Regresión Logística (*Logistic Regression*)

> La **Regresión Logística** es el modelo base por excelencia para clasificación probabilística. A diferencia de la regresión lineal (que predice valores $-\infty < y < \infty$), la regresión logística aplica la función sigmoide para mapear los resultados al intervalo $[0, 1]$, interpretándose como la **probabilidad de que una propiedad pertenezca a la categoría** `Premium`.
> * **Ventaja directiva:** Alta interpretabilidad a través de los coeficientes (`Odds Ratios`) para sustentar decisiones ante los directivos y el proceo de auditoría.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Modelo de Regresión Logística con regularización estándar
modelo_logistico = LogisticRegression(max_iter=1000, random_state=42)
modelo_logistico.fit(X_train_scaled, y_train)

# Inferencia de clases y probabilidades
y_pred_log = modelo_logistico.predict(X_test_scaled)
y_prob_log = modelo_logistico.predict_proba(X_test_scaled)[:, 1]

auc_log = roc_auc_score(y_test, y_prob_log)
print(f"Exactitud Global: {modelo_logistico.score(X_test_scaled, y_test):.2%}")
print(f"Área bajo la curva ROC (Capacidad de discriminación AUC): {auc_log:.2%}\n")
print("Reporte Ejecutivo de Regresión Logística:")
print(classification_report(y_test, y_pred_log, target_names=['Estándar', 'Premium']))

## 7. Clasificadores de Alto Desempeño para Datos Inmobiliarios

> ¿Qué otros clasificadores son adecuados para House Prices?
- Los datos inmobiliarios y transaccionales presentan relaciones no lineales complejas, interacciones entre atributos (por ejemplo: la calidad de acabados potencia el valor del área construida) y las distribuciones sesgadas. Para este tipo de datos tabulares, los clasificadores más idóneos son los **algoritmos de ensamble basados en árboles**:

> 1. **`Random Forest Classifier`:** Ensamble de múltiples árboles de decisión mediante *bagging*. Reduce la varianza y es altamente robusto ante variables irrelevantes o correlacionadas.
> 2. **`Gradient Boosting / HistGradientBoosting`:** Construye árboles de forma secuencial corrigiendo los errores residuales del árbol previo (`boosting`). Es el referente industrial en competencias de analítica y plataformas de producción financiera.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

# 1. Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)  # Árboles no requieren escalamiento obligatorio
y_pred_rf = rf_clf.predict(X_test)
y_prob_rf = rf_clf.predict_proba(X_test)[:, 1]

# 2. HistGradientBoosting Classifier
gb_clf = HistGradientBoostingClassifier(max_iter=150, random_state=42)
gb_clf.fit(X_train, y_train)
y_pred_gb = gb_clf.predict(X_test)
y_prob_gb = gb_clf.predict_proba(X_test)[:, 1]

print("Comparación de evaluación de desempeño de los Clasificadores")
print(f"1. Regresión Logística -> Accuracy: {modelo_logistico.score(X_test_scaled, y_test):.2%} | ROC-AUC: {auc_log:.2%}")
print(f"2. Random Forest -> Accuracy: {rf_clf.score(X_test, y_test):.2%} | ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.2%}")
print(f"3. Gradient Boosting -> Accuracy: {gb_clf.score(X_test, y_test):.2%} | ROC-AUC: {roc_auc_score(y_test, y_prob_gb):.2%}")

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# 1. Cálculo de curvas para cada clasificador
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
auc_log = roc_auc_score(y_test, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)
fpr_gb, tpr_gb, _ = roc_curve(y_test, y_prob_gb)
auc_gb = roc_auc_score(y_test, y_prob_gb)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=100, sharey=True)
modelos_info = [
    ('Regresión Logística', fpr_log, tpr_log, auc_log, '#1f77b4'),
    ('Random Forest',        fpr_rf,  tpr_rf,  auc_rf,  '#2ca02c'),
    ('Gradient Boosting',    fpr_gb,  tpr_gb,  auc_gb,  '#ff7f0e')
]

for ax, (nombre, fpr, tpr, auc, color) in zip(axes, modelos_info):
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'AUC = {auc:.2%}')
    ax.fill_between(fpr, tpr, alpha=0.15, color=color)
    ax.plot([0, 1], [0, 1], color='#7f7f7f', lw=1.2, linestyle='--', label='Azar (50%)')
    ax.set_title(nombre, fontsize=12, fontweight='bold')
    ax.set_xlabel('Tasa de Falsos Positivos (FPR)', fontsize=10)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower right', fontsize=10)

axes[0].set_ylabel('Tasa de Verdaderos Positivos (TPR)', fontsize=11, fontweight='bold')
plt.suptitle('Curvas ROC Individuales por Modelo Predictivo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 8. Diagnóstico de Gobierno Analítico y Relevancia de Variables

> Para sustentar las inversiones en modernización inmobiliaria o diseño de productos hipotecarios, **identificamos las variables (`Key Drivers`)** que más pesan en la clasificación del inmueble mediante la importancia de características del `Random Forest`.

In [ ]:
# Importancia de variables directivas en Random Forest
importancias = pd.Series(rf_clf.feature_importances_, index=all_features)
top_10_variables = importancias.sort_values(ascending=False).head(10)

print("Top 10 variables predictoras del Segmento Premium:")
for var, imp in top_10_variables.items():
    print(f"- {var:<20}: {imp:.2%}")

## 9. Inferencia y Simulación de Decisión Operativa en Producción

> Simulamos la evaluación automatizada de dos nuevas propiedades que ingresan al catálogo comercial de la organización, determinando tanto su **estimación continua en dólares (Regresión)** como su **probabilidad y categorización de segmento (Clasificación)**.

In [ ]:
# Simulación de 2 nuevos inmuebles evaluados por la plataforma transaccional
# Inmueble A: 12000 sqft, Calidad 6, Condición 6, 1200 sqft 1stFlr, 500 sqft 2ndFlr, 3 hab, 5 años, AC=1
# Inmueble B: 18000 sqft, Calidad 9, Condición 7, 2100 sqft 1stFlr, 1200 sqft 2ndFlr, 4 hab, 2 años, AC=1

nuevo_caso_A = np.zeros(len(all_features))
nuevo_caso_A[:8] = [12000, 6, 6, 1200, 500, 3, 5, 1]
if 'Nb_Timber' in all_features:
    nuevo_caso_A[all_features.index('Nb_Timber')] = 1

nuevo_caso_B = np.zeros(len(all_features))
nuevo_caso_B[:8] = [18000, 9, 7, 2100, 1200, 4, 2, 1]
if 'Nb_NoRidge' in all_features:
    nuevo_caso_B[all_features.index('Nb_NoRidge')] = 1

nuevos_inmuebles = np.array([nuevo_caso_A, nuevo_caso_B])

# 1. Inferencia Continua (Regresión Lineal Múltiple)
precios_estimados = modelo_regresion.predict(nuevos_inmuebles)

# 2. Inferencia Categórica y Probabilidad (Random Forest Classifier)
clases_predichas = rf_clf.predict(nuevos_inmuebles)
probabilidades = rf_clf.predict_proba(nuevos_inmuebles)[:, 1]

print("Reporte Ejecutivo de Inferencia en Tiempo Real")
for i, (p_est, c_pred, prob) in enumerate(zip(precios_estimados, clases_predichas, probabilidades), start=1):
    etiqueta = 'PREMIUM' if c_pred == 1 else 'ESTÁNDAR'
    print(f"Propiedad {i}:")
    print(f"  - Valoración estimada (Regresión): ${p_est:,.2f}")
    print(f"  - Clasificación de Portafolio:     {etiqueta}")
    print(f"  - Probabilidad de ser Premium:    {prob:.1%}")
    print(f"  - Recomendación Estratégica:      {'Asignar a fuerza comercial especializada' if c_pred == 1 else 'Comercializar por canal masivo digital'}\n")

> - La simulación de inferencia demuestra cómo los modelos automatizan el flujo de trabajo operacional en milisegundos con nuevos datos, lo cual mitiga la degradación de los modelos en producción `dift`.
> - La transformación digital se consolida cuando los modelos analíticos se integran en arquitecturas transaccionales en producción, reduciendo costos operativos y mejorando la experiencia del cliente final.

# Referencias

- Fuentes, A. (2018). *Become a Python Data Analyst*. Packt Publishing. ISBN 978-1-78728-430-2.
- Hwang, Y. y Burtch, N. (2024). *Machine Learning and Generative AI for Marketing*. Packt Publishing. ISBN 978-1-83588-940-4.
- Ping, D. (2024). *The Machine Learning Solutions Architect Handbook*. Packt Publishing. ISBN 978-1-80512-250-0.
- Reddi, J. (2025). *Machine Learning Systems*. School of Engineering and Applied Sciences Harvard University.